# Importation des modules

In [1]:
!pip install openpyxl
!pip install missingno
!pip install statsmodels


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import os
import statsmodels
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno

# Chargement des données
diviser les dépenses de santé par le PIB pour les avoir en %  
rajouter les données de dépenses de santé pour 1994 - 2004 en regardant sur les données de l'ocde  
rajouter la base mortalite  
faire le tri dans les pays de la bdd nb de médecins  


In [3]:
depenses_sante_vol = pd.read_excel(os.path.join("data", "Depenses_sante_en_volume_fr.xlsx"))
depenses_sante_PIB = pd.read_excel(os.path.join("data", "Depenses_Sante_PIB_OCDE_fr.xlsx"))
pib = pd.read_excel(os.path.join("data", "PIB_fr.xlsx"))
pib_par_habitant = pd.read_excel(os.path.join("data", "PIB_par_habitant_fr.xlsx"))
pop_tot = pd.read_excel(os.path.join("data", "Pop_tot_fr.xlsx"))
pop65 = pd.read_excel(os.path.join("data", "Pop+65ans_fr.xlsx"))
part_pop65 = pd.read_excel(os.path.join("data", "part_pop_plus_65_fr.xlsx"))
active_physicians = pd.read_excel(os.path.join("data", "active_physicians.xlsx"))
hospital_beds = pd.read_excel(os.path.join("data", "hospital_beds.xlsx"))
taux_deces_2ans = pd.read_excel('data/Taux_Deces_2ans.xlsx', index_col=None)
taux_deces_2ans = taux_deces_2ans.rename(columns={taux_deces_2ans.columns[0]: 'TIME'})
chom=pd.read_excel(os.path.join("data", "taux_chomage_panel.xlsx"))

# Mise en forme du panel

In [ ]:
def preparer_donnees(fichier, var) :   
    df = fichier.copy()
    df = df.replace(':', np.nan)
    df = pd.melt(df, id_vars=['TIME'], var_name='Year', value_name=var)
    df = df.rename(columns={'TIME': 'Country'})
    df['Year'] = df['Year'].astype(int)  # S'assurer que l'année est un entier
    df = df.set_index(['Country', 'Year']).sort_index()
    return df

In [8]:
panel_depenses_sante_PIB = preparer_donnees(depenses_sante_PIB, "Depenses de sante en % du PIB")
panel_PIB_par_habitant = preparer_donnees(pib_par_habitant, "PIB par habitant")
#panel_PIB = preparer_donnees(pib, "PIB")
panel_chom=preparer_donnees(chom, "Chomage")
panel = pd.merge(panel_depenses_sante_PIB, panel_PIB_par_habitant, on=['Country', 'Year'], how='outer')
#panel = pd.merge(panel_depenses_sante_PIB, panel_PIB, on=['Country', 'Year'], how='outer')
panel_pop65 = preparer_donnees(part_pop65, "Part des +65 ans")
panel = pd.merge(panel, panel_pop65, on=['Country', 'Year'], how='outer')
panel_active_physicians = preparer_donnees(active_physicians, "Practiciens pour 1000 habitants")
panel = pd.merge(panel, panel_active_physicians, on=['Country', 'Year'], how='outer')
panel_hospital_beds = preparer_donnees(hospital_beds, "Lits d hopitaux pour 1000 habitants")
panel = pd.merge(panel, panel_hospital_beds, on=['Country', 'Year'], how='outer')
panel_taux_deces_2ans = preparer_donnees(taux_deces_2ans, "Taux deces 2 ans")
panel = pd.merge(panel, panel_taux_deces_2ans, on=['Country', 'Year'], how='outer')
panel = pd.merge(panel, panel_chom, on=['Country', 'Year'], how='outer')
panel['Depenses t-1'] = panel.groupby(level='Country')['Depenses de sante en % du PIB'].shift(1)

In [9]:
annees_a_supprimer = [1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2024, 2025]
panel = panel.drop(annees_a_supprimer, level='Year')
panel = panel.drop("Suède", level='Country')
panel

Depenses de sante en % du PIB  PIB par habitant  \
Country   Year                                                    
Allemagne 2005                         10.107             129.3   
          2006                          9.962             128.1   
          2007                          9.835             127.2   
          2008                         10.032             126.3   
          2009                         10.970             127.9   
...                                       ...               ...   
Tchéquie  2019                          7.488              68.6   
          2020                          8.970              68.8   
          2021                          9.154              70.4   
          2022                          8.471              73.8   
          2023                          8.427              76.4   

                Part des +65 ans  Practiciens pour 1000 habitants  \
Country   Year                                                      
Allemagne 2005          0.186270                             3.40   
          2006          0.192509                             3.44   
          2007          0.198011                             3.49   
          2008          0.200914                             3.54   
          2009          0.204006                             3.62   
...                          ...                              ...   
Tchéquie  2019          0.195930                             4.07   
          2020          0.199331                             4.13   
          2021          0.205037                             4.26   
          2022          0.206254                             4.32   
          2023          0.203911                             4.86   

                Lits d hopitaux pour 1000 habitants  Taux deces 2 ans  \
Country   Year                                                          
Allemagne 2005                                 6.26             2.002   
          2006                                 6.13             2.000   
          2007                                 5.57             2.031   
          2008                                 6.10             2.066   
          2009                                 6.11             2.089   
...                                             ...               ...   
Tchéquie  2019                                 5.67             2.269   
          2020                                 5.86             2.517   
          2021                                 5.93             2.478   
          2022                                 4.92             2.216   
          2023                                 4.83               NaN   

                Chomage  Depenses t-1  
Country   Year                         
Allemagne 2005     11.2         9.975  
          2006     10.3        10.107  
          2007      8.7         9.962  
          2008      7.5         9.835  
          2009      7.8        10.032  
...                 ...           ...  
Tchéquie  2019      2.0         7.372  
          2020      2.6         7.488  
          2021      2.8         8.970  
          2022      2.2         9.154  
          2023      2.6         8.471  

[380 rows x 8 columns]

## Données manquantes

In [12]:
#gérer les données manquantes
panel_transformed = panel.copy()

for col in ['Depenses de sante en % du PIB', 'PIB par habitant', 'Part des +65 ans', 'Practiciens pour 1000 habitants',
       'Lits d hopitaux pour 1000 habitants', 'Taux deces 2 ans', 'Chomage'] :
    panel_transformed[col] = panel_transformed.groupby('Country')[col].transform(lambda x: x.fillna(x.median()))

In [14]:
panel_transformed.to_csv("panelV2.csv", index=True)

# Régression TTD  
On veut vérifier si contrôler par le TTD atténue l'impact du vieillissement sur les dépenses de santé.  
On fait notre régression sur plusieurs tranches d'âge au-dessus de 65 ans. Comme TTD, on utilise l'espérance de vie résiduelle de la tranche d'âge sur laquelle on se place et on remplace

# Two Way fixed effects

In [38]:
print(panel.index.names)
print(panel.columns.tolist())


['Country', 'Year']
['Depenses de sante en % du PIB', 'PIB par habitant', 'Part des +65 ans', 'Practiciens pour 1000 habitants', 'Lits d hopitaux pour 1000 habitants', 'Taux deces 2 ans', 'Chomage', 'Depenses t-1']


In [40]:
import pandas as pd
import numpy as np
from scipy import stats

DEP     = 'Depenses de sante en % du PIB'
M1_REGS = ['Part des +65 ans', 'PIB par habitant', 'Chomage']
M2_REGS = ['Part des +65 ans', 'Taux deces 2 ans', 'PIB par habitant', 'Chomage']

def twfe(data, regressors, dep):
    cols = ['Country', 'Year', dep] + regressors
    sub  = data[cols].dropna().copy().sort_values(['Country', 'Year'])
    for col in [dep] + regressors:
        sub[col+'_fe'] = (sub[col]
                         - sub.groupby('Year')[col].transform('mean')
                         - sub.groupby('Country')[col].transform('mean')
                         + sub[col].mean())
    Y         = sub[dep+'_fe'].values
    X         = sub[[r+'_fe' for r in regressors]].values
    countries = sub['Country'].values
    N, k      = X.shape
    G         = len(np.unique(countries))
    beta  = np.linalg.lstsq(X, Y, rcond=None)[0]
    resid = Y - X @ beta
    meat = np.zeros((k, k))
    for c in np.unique(countries):
        idx = countries == c
        sc  = X[idx].T @ resid[idx]
        meat += np.outer(sc, sc)
    corr    = G/(G-1) * (N-1)/(N-k)
    XtX_inv = np.linalg.inv(X.T @ X)
    se      = np.sqrt(np.diag(corr * XtX_inv @ meat @ XtX_inv))
    t       = beta / se
    p       = 2 * (1 - stats.t.cdf(np.abs(t), df=G-1))
    r2      = 1 - np.sum(resid**2) / np.sum((Y - Y.mean())**2)
    return beta, se, t, p, r2, N

def sig(p):
    return "***" if p<0.001 else ("**" if p<0.01 else ("*" if p<0.05 else "ns"))

panel = panel.reset_index()
b1, se1, t1, p1, r2_1, n1 = twfe(panel, M1_REGS, DEP)
b2, se2, t2, p2, r2_2, n2 = twfe(panel, M2_REGS, DEP)

all_vars = ['Part des +65 ans', 'Taux deces 2 ans', 'PIB par habitant', 'Chomage']
labels   = ['Part des +65 ans', 'Taux de décès <2 ans', 'PIB par habitant', 'Taux de chômage']

print(f"\n{'='*70}")
print(f"  TWFE — Dépenses santé (% PIB)")
print(f"{'='*70}")
print(f"  {'Variable':<28} {'Modèle 1':>18} {'Modèle 2':>18}")
print(f"  {'-'*68}")

for var, lab in zip(all_vars, labels):
    if var in M1_REGS:
        i1 = M1_REGS.index(var)
        c1 = f"{b1[i1]:.4f} {sig(p1[i1])}"
        s1 = f"({se1[i1]:.4f})"
        pv1 = f"[p={p1[i1]:.3f}]"
    else:
        c1, s1, pv1 = "—", "", ""

    i2  = M2_REGS.index(var)
    c2  = f"{b2[i2]:.4f} {sig(p2[i2])}"
    s2  = f"({se2[i2]:.4f})"
    pv2 = f"[p={p2[i2]:.3f}]"

    print(f"  {lab:<28} {c1:>18} {c2:>18}")
    print(f"  {'':28} {s1:>18} {s2:>18}")
    print(f"  {'':28} {pv1:>18} {pv2:>18}")
    print()

print(f"  {'-'*68}")
print(f"  {'FE pays + année':<28} {'Oui':>18} {'Oui':>18}")
print(f"  {'Observations':<28} {n1:>18} {n2:>18}")
print(f"  {'R² within':<28} {r2_1:>18.4f} {r2_2:>18.4f}")
print(f"{'='*70}")
print("  SE clustérisées par pays. * p<0.05  ** p<0.01  *** p<0.001")


  TWFE — Dépenses santé (% PIB)
  Variable                               Modèle 1           Modèle 2
  --------------------------------------------------------------------
  Part des +65 ans                     -2.6001 ns          0.0449 ns
                                        (14.6813)          (14.3522)
                                        [p=0.861]          [p=0.998]

  Taux de décès <2 ans                          —         -0.2347 ns
                                                            (1.2145)
                                                           [p=0.849]

  PIB par habitant                     -0.0296 **        -0.0331 ***
                                         (0.0096)           (0.0082)
                                        [p=0.006]          [p=0.001]

  Taux de chômage                       -0.0472 *          -0.0506 *
                                         (0.0215)           (0.0234)
                                        [p=0.041]          [p=0.0

In [41]:
def ols_fe_year(data, regressors, dep):
    cols = ['Country', 'Year', dep] + regressors
    sub  = data[cols].dropna().copy().sort_values(['Country', 'Year'])
    
    # Déméaning uniquement sur l'année (effets fixes année seulement)
    for col in [dep] + regressors:
        sub[col+'_fe'] = (sub[col]
                         - sub.groupby('Year')[col].transform('mean')
                         + sub[col].mean())
    
    Y         = sub[dep+'_fe'].values
    X         = sub[[r+'_fe' for r in regressors]].values
    countries = sub['Country'].values
    N, k      = X.shape
    G         = len(np.unique(countries))
    
    beta  = np.linalg.lstsq(X, Y, rcond=None)[0]
    resid = Y - X @ beta
    
    # Clustering par pays
    meat = np.zeros((k, k))
    for c in np.unique(countries):
        idx = countries == c
        sc  = X[idx].T @ resid[idx]
        meat += np.outer(sc, sc)
    
    corr    = G/(G-1) * (N-1)/(N-k)
    XtX_inv = np.linalg.inv(X.T @ X)
    se      = np.sqrt(np.diag(corr * XtX_inv @ meat @ XtX_inv))
    t       = beta / se
    p       = 2 * (1 - stats.t.cdf(np.abs(t), df=G-1))
    r2      = 1 - np.sum(resid**2) / np.sum((Y - Y.mean())**2)
    
    return beta, se, t, p, r2, N


b1_y, se1_y, t1_y, p1_y, r2_1_y, n1_y = ols_fe_year(panel, M1_REGS, DEP)
b2_y, se2_y, t2_y, p2_y, r2_2_y, n2_y = ols_fe_year(panel, M2_REGS, DEP)

print(f"\n{'='*70}")
print(f"  OLS — FE année uniquement (sans FE pays)")
print(f"{'='*70}")
print(f"  {'Variable':<28} {'Modèle 1':>18} {'Modèle 2':>18}")
print(f"  {'-'*68}")

for var, lab in zip(all_vars, labels):
    if var in M1_REGS:
        i1 = M1_REGS.index(var)
        c1  = f"{b1_y[i1]:.4f} {sig(p1_y[i1])}"
        s1  = f"({se1_y[i1]:.4f})"
        pv1 = f"[p={p1_y[i1]:.3f}]"
    else:
        c1, s1, pv1 = "—", "", ""

    i2  = M2_REGS.index(var)
    c2  = f"{b2_y[i2]:.4f} {sig(p2_y[i2])}"
    s2  = f"({se2_y[i2]:.4f})"
    pv2 = f"[p={p2_y[i2]:.3f}]"

    print(f"  {lab:<28} {c1:>18} {c2:>18}")
    print(f"  {'':28} {s1:>18} {s2:>18}")
    print(f"  {'':28} {pv1:>18} {pv2:>18}")
    print()

print(f"  {'-'*68}")
print(f"  {'FE pays':<28} {'Non':>18} {'Non':>18}")
print(f"  {'FE année':<28} {'Oui':>18} {'Oui':>18}")
print(f"  {'Observations':<28} {n1_y:>18} {n2_y:>18}")
print(f"  {'R² within':<28} {r2_1_y:>18.4f} {r2_2_y:>18.4f}")
print(f"{'='*70}")
print("  SE clustérisées par pays. * p<0.05  ** p<0.01  *** p<0.001")


  OLS — FE année uniquement (sans FE pays)
  Variable                               Modèle 1           Modèle 2
  --------------------------------------------------------------------
  Part des +65 ans                    41.6593 ***        68.2184 ***
                                         (4.7360)          (13.0514)
                                        [p=0.000]          [p=0.000]

  Taux de décès <2 ans                          —          -2.0101 *
                                                            (0.7883)
                                                           [p=0.020]

  PIB par habitant                      0.0081 ns          0.0040 ns
                                         (0.0070)           (0.0065)
                                        [p=0.260]          [p=0.547]

  Taux de chômage                       0.0148 ns          0.0021 ns
                                         (0.0468)           (0.0515)
                                        [p=0.756]     